# ThyroCheck AI — train FibonacciNet on a Kaggle GPU

**Before running:**
1. *Add Input* → search **Algerian Ultrasound Images Thyroid Dataset (AUITD)** → add it.
2. Upload `thyrocheck-ai.zip` as your own Kaggle dataset (e.g. `thyrocheck-ai-code`) and add it as input too.
3. *Settings* → *Accelerator* → **GPU T4 x2** (or P100).
4. *Run All*. Download `/kaggle/working/thyrocheck-ai/outputs` at the end.

In [ ]:
import glob, os, shutil, subprocess
# find the uploaded project (zip extracted by Kaggle, or the zip itself)
cands = glob.glob('/kaggle/input/**/run_pipeline.sh', recursive=True)
if cands:
    shutil.copytree(os.path.dirname(cands[0]), '/kaggle/working/thyrocheck-ai', dirs_exist_ok=True)
else:
    z = glob.glob('/kaggle/input/**/thyrocheck-ai*.zip', recursive=True)[0]
    shutil.unpack_archive(z, '/kaggle/working')
%cd /kaggle/working/thyrocheck-ai
!ls

In [ ]:
# Kaggle already has TensorFlow; add the few extras
!pip install -q python-docx
import sys; sys.path.insert(0, 'src')
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# Link the attached Kaggle dataset into data/raw (no download needed on Kaggle)
data_dirs = [d for d in glob.glob('/kaggle/input/*') if 'thyroid' in d.lower() or 'auitd' in d.lower()]
print(data_dirs)
os.makedirs('data', exist_ok=True)
if os.path.lexists('data/raw'): os.remove('data/raw') if os.path.islink('data/raw') else shutil.rmtree('data/raw')
os.symlink(data_dirs[0], 'data/raw')
!find data/raw/ -maxdepth 3 -type d | head -30

In [ ]:
!PYTHONPATH=src python -m thyrocheck.data.prepare

If `prepare` skipped images, add the printed folder names to `data.label_keywords` in `configs/config.yaml` and rerun the cell above.

For benign-vs-malignant only, add `--set "data.classes=[benign, malignant]"` to every command.

In [ ]:
!PYTHONPATH=src python -m thyrocheck.training.train

In [ ]:
!PYTHONPATH=src python -m thyrocheck.evaluation.evaluate
from IPython.display import Image, display
for f in ['training_curves', 'confusion_matrix', 'roc_curves']:
    display(Image(f'outputs/figures/{f}.png', width=520))

In [ ]:
!PYTHONPATH=src python -m thyrocheck.xai.gradcam --n 9
for f in sorted(glob.glob('outputs/gradcam/*.png'))[:6]:
    display(Image(f, width=520))

In [ ]:
# Ablation: same training without the parallel concat blocks
# !PYTHONPATH=src python -m thyrocheck.training.train --set model.use_pcb_2_4=false model.use_pcb_3_5=false train.model_path=outputs/models/no_pcb.keras

In [ ]:
# Package results to download
shutil.make_archive('/kaggle/working/thyrocheck_outputs', 'zip', 'outputs')
print('Download /kaggle/working/thyrocheck_outputs.zip from the Output panel')